# C8-embeddings — Practice p16 — Solution

In [ ]:
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
import gensim.downloader

In [ ]:
kv = gensim.downloader.load("glove-wiki-gigaword-100")

WORDS = ["anchor", "mast", "rudder", "sail", "deck", "cabin"]
V = np.asarray(kv[WORDS], dtype=np.float64)
W = V / np.sqrt((V * V).sum(axis=1, keepdims=True))
S = W @ W.T


def neighbors_bad(S, i, k):
    return np.argsort(S[i])[:k]


bad_idx = neighbors_bad(S, 0, 3)
bad_words = np.asarray(WORDS)[bad_idx].tolist()
bad_sims = S[0, bad_idx]

descending_unfiltered = np.argsort(S[0])[::-1]
stillbad_words = np.asarray(WORDS)[descending_unfiltered[:3]].tolist()


def neighbors_fixed(S, i, k):
    order = np.argsort(S[i])[::-1]
    return order[order != i][:k]


good_idx = neighbors_fixed(S, 0, 3)
good_words = np.asarray(WORDS)[good_idx].tolist()
good_sims = S[0, good_idx]

### Prediction and diagnosis

Ascending sorting returns the three *least* similar candidates: `rudder`, `sail`, and `cabin`, with similarities about 0.2805, 0.2984, and 0.3140. Reversing alone fixes direction but leaves the second bug: the query itself, with similarity 1, remains first and displaces a genuine neighbor.

### Large-list caution

Frequency effects can make a very common word topically uninformative, while hubness can place a central word in many neighbor lists. Treat ranks as evidence to inspect, checking the similarity values and candidate universe rather than trusting rank alone.

### Answer check

In [ ]:
assert bad_words == ["rudder", "sail", "cabin"]
assert np.allclose(bad_sims,
                   np.array([0.28052437426714055, 0.2984027799314129,
                             0.31396072636893546]), atol=1e-12, rtol=0)
assert stillbad_words == ["anchor", "deck", "mast"]
assert good_words == ["deck", "mast", "cabin"]
assert np.allclose(good_sims,
                   np.array([0.40676214654644083, 0.3645668221931425,
                             0.31396072636893546]), atol=1e-12, rtol=0)
assert 0 not in good_idx